<a href="https://colab.research.google.com/github/NubasVAdidas/potapov_1iis_41/blob/%D0%91%D0%BE%D0%BB%D1%8C%D1%88%D0%B8%D0%B5-%D1%8F%D0%B7%D1%8B%D0%BA%D0%BE%D0%B2%D1%8B%D0%B5-%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D0%B8/notebooks/lr_3_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ЛАБОРАТОРНАЯ РАБОТА №3: NLP**

# ЗАДАНИЯ ДЛЯ ПОДГОТОВКИ С РЕШЕНИЯМИ ИЗ МЕТОДИЧКИ

In [184]:
!pip install spacy
!python -m spacy download ru_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 14.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


**Загрузка библиотек**

In [185]:
import re
import math
import spacy
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nlp = spacy.load("ru_core_news_sm")
print("\n\nComplete!")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!




Complete!


**Исходные данные**

In [186]:
corpus = [
    "Сегодня была отличная погода, и мы пошли гулять в красивый парк!",
    "Ужасная погода... Весь день идёт сильный дождь, прогулка отменяется.",
    "В парке мы видели красивых птиц. Погода шепчет, солнце светит ярко.",
    "Дождь не прекращается! Сильный ветер сорвал все планы на вечер."
]

sample_text = corpus[1]
print(f"Иcходный текст для анализов: {sample_text}")

Иcходный текст для анализов: Ужасная погода... Весь день идёт сильный дождь, прогулка отменяется.


**RegEx**

In [187]:
text_lowercase = sample_text.lower()
text_cleaned = re.sub(r'[^а-яё\s]', '', text_lowercase)
print(f"Текст после очистки RegEx: \n{text_cleaned}")

Текст после очистки RegEx: 
ужасная погода весь день идёт сильный дождь прогулка отменяется


**Токенизация, удаление стоп-слов и Лемматизация**

In [188]:
russian_stopwords = set(stopwords.words('russian'))
doc = nlp(text_cleaned)
filtered_tokens = []
for token in doc:
  lemma = token.lemma_
  if lemma not in russian_stopwords and lemma.strip():
    filtered_tokens.append(lemma)
print(f"Токены после лемматизации и удаления стоп-слов:\n{filtered_tokens}")

Токены после лемматизации и удаления стоп-слов:
['ужасный', 'погода', 'весь', 'день', 'идти', 'сильный', 'дождь', 'прогулка', 'отменяться']


**Полная предобработка**

In [189]:
def pipeline_preprocess(text):
  text = text.lower()
  text = re.sub(r'[^а-яё\s]', '', text)
  doc = nlp(text)
  return [lemma for token in doc if (lemma := token.lemma_.strip()) and lemma not in russian_stopwords]
processed_corpus = [pipeline_preprocess(text) for text in corpus]

for i, doc_tokens in enumerate(processed_corpus):
  print(f"Документ {i+1}: {doc_tokens}")


Документ 1: ['сегодня', 'отличный', 'погода', 'пойти', 'гулять', 'красивый', 'парк']
Документ 2: ['ужасный', 'погода', 'весь', 'день', 'идти', 'сильный', 'дождь', 'прогулка', 'отменяться']
Документ 3: ['парк', 'видеть', 'красивый', 'птица', 'погода', 'шепчет', 'солнце', 'светить', 'ярко']
Документ 4: ['дождь', 'прекращаться', 'сильный', 'ветер', 'сорвать', 'весь', 'план', 'вечер']


**Векторизация текста**

In [190]:
def vocab():
  vocabulary = sorted(list(set([word for doc in processed_corpus for word in doc])))
  print(f"Уникальный словарь корпуса (всего слов: {len(vocabulary)}): {vocabulary}")
vocab()
def calculate_tfidf(word, current_doc, all_docs):
  tf = current_doc.count(word) / len(current_doc) if len(current_doc) > 0 else 0

  docs_with_word = sum(1 for d in all_docs if word in d)
  idf = math.log(len(all_docs) / docs_with_word)
  return round(tf * idf, 4)

print("\nВекторы документов (TF-IDF числовые представления):")
def calc_vector():
  for i, doc in enumerate(processed_corpus):
    vector = []
    for word in vocabulary:
      tfidf_value = calculate_tfidf(word, doc, processed_corpus)
      vector.append(tfidf_value)
    print(f"Вектор Документа {i+1}: {vector}")
calc_vector()

Уникальный словарь корпуса (всего слов: 26): ['весь', 'ветер', 'вечер', 'видеть', 'гулять', 'день', 'дождь', 'идти', 'красивый', 'отличный', 'отменяться', 'парк', 'план', 'погода', 'пойти', 'прекращаться', 'прогулка', 'птица', 'светить', 'сегодня', 'сильный', 'солнце', 'сорвать', 'ужасный', 'шепчет', 'ярко']

Векторы документов (TF-IDF числовые представления):
Вектор Документа 1: [0.0, 0.0, 0.0, 0.0, 0.198, 0.0, 0.0, 0.0, 0.099, 0.198, 0.0, 0.099, 0.0, 0.0411, 0.198, 0.0, 0.0, 0.0, 0.0, 0.198, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Вектор Документа 2: [0.077, 0.0, 0.0, 0.0, 0.0, 0.154, 0.077, 0.154, 0.0, 0.0, 0.154, 0.0, 0.0, 0.032, 0.0, 0.0, 0.154, 0.0, 0.0, 0.0, 0.077, 0.0, 0.0, 0.154, 0.0, 0.0]
Вектор Документа 3: [0.0, 0.0, 0.0, 0.154, 0.0, 0.0, 0.0, 0.0, 0.077, 0.0, 0.0, 0.077, 0.0, 0.032, 0.0, 0.0, 0.0, 0.154, 0.154, 0.0, 0.0, 0.154, 0.0, 0.0, 0.154, 0.154]
Вектор Документа 4: [0.0866, 0.1733, 0.1733, 0.0, 0.0, 0.0, 0.0866, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1733, 0.0, 0.0, 0.1733, 0.0, 0.0, 0.0,

# ЗАДАНИЯ ДЛЯ САМОСТОЯТЕЛЬНОЙ РАБОТЫ

Задание 1. Глубокий анализ и кастомизация стоп-слов

In [191]:
def stopwords_count(current_words):
  count_stopwords = len(current_words)
  print(f"Кол-во русских стоп-слов: {count_stopwords}")
  sort_stopwords = sorted(current_words)
  print(f"Стоп-слова:\n{sort_stopwords}\n")

stopwords_count(russian_stopwords)
new_stopwords = russian_stopwords | {'сегодня', 'весь'}
print("Добавлены слова")
stopwords_count(new_stopwords)

def pipeline_preprocess_temp(text, stopword):
  text = text.lower()
  text = re.sub(r'[^а-яё\s]', '', text)
  doc = nlp(text)
  return [lemma for token in doc if (lemma := token.lemma_.strip()) and lemma not in stopword]
processed_corpus_temp = [pipeline_preprocess_temp(text, new_stopwords) for text in corpus]

for i, doc_tokens in enumerate(processed_corpus_temp):
  print(f"Документ {i+1}: {doc_tokens}")

Кол-во русских стоп-слов: 151
Стоп-слова:
['а', 'без', 'более', 'больше', 'будет', 'будто', 'бы', 'был', 'была', 'были', 'было', 'быть', 'в', 'вам', 'вас', 'вдруг', 'ведь', 'во', 'вот', 'впрочем', 'все', 'всегда', 'всего', 'всех', 'всю', 'вы', 'где', 'да', 'даже', 'два', 'для', 'до', 'другой', 'его', 'ее', 'ей', 'ему', 'если', 'есть', 'еще', 'ж', 'же', 'за', 'зачем', 'здесь', 'и', 'из', 'или', 'им', 'иногда', 'их', 'к', 'как', 'какая', 'какой', 'когда', 'конечно', 'кто', 'куда', 'ли', 'лучше', 'между', 'меня', 'мне', 'много', 'может', 'можно', 'мой', 'моя', 'мы', 'на', 'над', 'надо', 'наконец', 'нас', 'не', 'него', 'нее', 'ней', 'нельзя', 'нет', 'ни', 'нибудь', 'никогда', 'ним', 'них', 'ничего', 'но', 'ну', 'о', 'об', 'один', 'он', 'она', 'они', 'опять', 'от', 'перед', 'по', 'под', 'после', 'потом', 'потому', 'почти', 'при', 'про', 'раз', 'разве', 'с', 'сам', 'свою', 'себе', 'себя', 'сейчас', 'со', 'совсем', 'так', 'такой', 'там', 'тебя', 'тем', 'теперь', 'то', 'тогда', 'того', 'тоже',

**Задание 2. Исследование масштабируемости корпуса и динамики
векторов**

In [192]:
vocab()
for i, doc_tokens in enumerate(processed_corpus):
  print(f"Документ {i+1}: {doc_tokens}")
calc_vector()
corpus.append("На улице поют птицы, погода прекрасная! Поэтому сегодня мы гуляем")
processed_corpus = [pipeline_preprocess(text) for text in corpus]
print("Добавлен новый документ\n")
vocab()
for i, doc_tokens in enumerate(processed_corpus):
  print(f"Документ {i+1}: {doc_tokens}")
calc_vector()



Уникальный словарь корпуса (всего слов: 26): ['весь', 'ветер', 'вечер', 'видеть', 'гулять', 'день', 'дождь', 'идти', 'красивый', 'отличный', 'отменяться', 'парк', 'план', 'погода', 'пойти', 'прекращаться', 'прогулка', 'птица', 'светить', 'сегодня', 'сильный', 'солнце', 'сорвать', 'ужасный', 'шепчет', 'ярко']
Документ 1: ['сегодня', 'отличный', 'погода', 'пойти', 'гулять', 'красивый', 'парк']
Документ 2: ['ужасный', 'погода', 'весь', 'день', 'идти', 'сильный', 'дождь', 'прогулка', 'отменяться']
Документ 3: ['парк', 'видеть', 'красивый', 'птица', 'погода', 'шепчет', 'солнце', 'светить', 'ярко']
Документ 4: ['дождь', 'прекращаться', 'сильный', 'ветер', 'сорвать', 'весь', 'план', 'вечер']
Вектор Документа 1: [0.0, 0.0, 0.0, 0.0, 0.198, 0.0, 0.0, 0.0, 0.099, 0.198, 0.0, 0.099, 0.0, 0.0411, 0.198, 0.0, 0.0, 0.0, 0.0, 0.198, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Вектор Документа 2: [0.077, 0.0, 0.0, 0.0, 0.0, 0.154, 0.077, 0.154, 0.0, 0.0, 0.154, 0.0, 0.0, 0.032, 0.0, 0.0, 0.154, 0.0, 0.0, 0.0, 0.077

**Задание 3. Модификация регулярных выражений под сохранение
числовых данных**


In [193]:
corpus_with_numbers = corpus.copy()
corpus_with_numbers.append("В 2026 году мы выполнили 3 проекта и обработали 15 документов!")

def pipeline_preprocess_with_numbers(text):
    text = text.lower()
    text = re.sub(r'[^а-яё0-9\s]', '', text)
    doc = nlp(text)
    return [lemma for token in doc if (lemma := token.lemma_.strip()) and lemma not in russian_stopwords]

processed_corpus_numbers = [pipeline_preprocess_with_numbers(text) for text in corpus_with_numbers]
print(f"\nИсходный текст: {corpus_with_numbers}")
for i, doc_tokens in enumerate(processed_corpus_numbers):
  print(f"Документ {i+1}: {doc_tokens}")


Исходный текст: ['Сегодня была отличная погода, и мы пошли гулять в красивый парк!', 'Ужасная погода... Весь день идёт сильный дождь, прогулка отменяется.', 'В парке мы видели красивых птиц. Погода шепчет, солнце светит ярко.', 'Дождь не прекращается! Сильный ветер сорвал все планы на вечер.', 'На улице поют птицы, погода прекрасная! Поэтому сегодня мы гуляем', 'В 2026 году мы выполнили 3 проекта и обработали 15 документов!']
Документ 1: ['сегодня', 'отличный', 'погода', 'пойти', 'гулять', 'красивый', 'парк']
Документ 2: ['ужасный', 'погода', 'весь', 'день', 'идти', 'сильный', 'дождь', 'прогулка', 'отменяться']
Документ 3: ['парк', 'видеть', 'красивый', 'птица', 'погода', 'шепчет', 'солнце', 'светить', 'ярко']
Документ 4: ['дождь', 'прекращаться', 'сильный', 'ветер', 'сорвать', 'весь', 'план', 'вечер']
Документ 5: ['улица', 'петь', 'птица', 'погода', 'прекрасный', 'поэтому', 'сегодня', 'гулять']
Документ 6: ['2026', 'год', 'выполнить', '3', 'проект', 'обработать', '15', 'документ']
